# QUELL Step 05 — Buyuk model merdiveni (4-bit QLoRA, aile-farkindalikli)

Blok 1'de **MODEL** ve **DATASET**'i degistir, sirayla Blok 1->2->3 calistir. gpt2=fp32, digerleri=4-bit QLoRA. Bos GPU otomatik secilir. Test buyukse stratified subset ile degerlendirir.

Ilk: Qwen/Qwen2.5-1.5B + edge_iiotset (acik model, token yok). KIYAS rowsini paylas.

In [ ]:
# ===== BLOCK 1/3: preparation (family-aware: gpt2=fp32, others=4-bit QLoRA) =====
import os, subprocess
try:
    _o=subprocess.check_output("nvidia-smi --query-gpu=index,memory.free --format=csv,noheader,nounits",shell=True,text=True)
    _f=[(int(x.split(",")[0]),int(x.split(",")[1])) for x in _o.strip().splitlines()]
    _b=max(_f,key=lambda t:t[1]); os.environ["CUDA_VISIBLE_DEVICES"]=str(_b[0])
    os.environ["PYTORCH_CUDA_ALLOC_CONF"]="expandable_segments:True"
    print("GPU free memory (MiB):",_f," -> selected GPU:",_b[0],flush=True)
except Exception as e: print("GPU secim atlandi:",e)
import json, time, sys, glob
from pathlib import Path
import numpy as np, pandas as pd
from pandas.api.types import is_numeric_dtype
for pk in ["transformers","peft","accelerate","bitsandbytes"]:
    try: __import__(pk)
    except Exception: subprocess.run([sys.executable,"-m","pip","install","-q",pk])
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from sklearn.preprocessing import LabelEncoder

# ===== CHANGE ONLY THESE TWO =====
MODEL   = "Qwen/Qwen2.5-1.5B"     # later: meta-llama/Llama-3.2-1B, mistralai/Mistral-7B-v0.3, meta-llama/Meta-Llama-3-8B
DATASET = "edge_iiotset"          # edge_iiotset / ciciot2023 / nbaiot
# ======================================
TRAIN_CAP=2000; EPOCHS=2; SEED=42; EVAL_CAP=60000
torch.manual_seed(SEED); np.random.seed(SEED)
ROOT=Path.home()/"quell-edge-llm-ids"; PROC=ROOT/"data"/"processed"; SPL=ROOT/"splits"; RES=ROOT/"results"
rep=json.load(open(RES/"split_report.json")); meta=rep[DATASET]
label=meta["label_col"]; group=meta.get("group_col"); tcol=meta.get("time_col")
LABELISH={"label","attack","attack_type","attack_label","type","class","category","marker","__label__"}
df=pd.read_parquet(PROC/f"{DATASET}.parquet").reset_index(drop=True)
sp=np.load(SPL/f"{DATASET}_split.npz"); tr_idx,te_idx=sp["train"],sp["test"]
drop=set([label])|set(meta.get("leaky_candidates",[]))
if group: drop.add(group)
if tcol: drop.add(tcol)
for c in df.columns:
    if c!=label and c.lower() in LABELISH: drop.add(c)
feats=[c for c in df.columns if c not in drop]
def row_to_text(r):
    parts=[]
    for c in feats:
        v=r[c]
        if isinstance(v,(float,np.floating)): v=round(float(v),4)
        parts.append(f"{c}={v}")
    return "Network traffic flow. " + ", ".join(parts) + " . Attack type:"
print("feature->text ("+str(len(df))+" rows)...",flush=True)
texts_all=df.apply(row_to_text,axis=1).values; y_all=df[label].astype(str).values
rng=np.random.default_rng(SEED); tr_sel=[]
for cls in pd.unique(y_all[tr_idx]):
    ids=tr_idx[y_all[tr_idx]==cls]
    if len(ids)>TRAIN_CAP: ids=rng.choice(ids,TRAIN_CAP,replace=False)
    tr_sel+=ids.tolist()
tr_sel=np.array(sorted(tr_sel))
le=LabelEncoder().fit(y_all[tr_sel]); K=len(le.classes_); classes_all=sorted(pd.unique(y_all).tolist())
MAX_LEN = 256 if len(feats)<=64 else 512
print(f"MODEL={MODEL} | {DATASET}: feature={len(feats)} MAX_LEN={MAX_LEN} train={len(tr_sel):,} FULLtest={len(te_idx):,} class={K}",flush=True)

tok=AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None: tok.pad_token=tok.eos_token
class DS(torch.utils.data.Dataset):
    def __init__(self,idx): self.idx=idx
    def __len__(self): return len(self.idx)
    def __getitem__(self,i):
        j=self.idx[i]; enc=tok(texts_all[j],truncation=True,max_length=MAX_LEN,padding="max_length",return_tensors="pt")
        it={k:v.squeeze(0) for k,v in enc.items()}; it["labels"]=torch.tensor(int(le.transform([y_all[j]])[0])); return it

is_gpt2="gpt2" in MODEL.lower()
BF16=torch.cuda.is_available() and torch.cuda.is_bf16_supported()
if is_gpt2:
    model=AutoModelForSequenceClassification.from_pretrained(MODEL,num_labels=K); targets=["c_attn"]
else:
    bnb=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,bnb_4bit_use_double_quant=True)
    model=AutoModelForSequenceClassification.from_pretrained(MODEL,num_labels=K,
        quantization_config=bnb,device_map={"":0});
    model=prepare_model_for_kbit_training(model); targets=["q_proj","v_proj"]
model.config.pad_token_id=tok.pad_token_id
model=get_peft_model(model,LoraConfig(task_type=TaskType.SEQ_CLS,r=16,lora_alpha=32,
    lora_dropout=0.05,target_modules=targets,modules_to_save=["score"]))
model.print_trainable_parameters()
BIG=any(s in MODEL.lower() for s in ["7b","8b"])
print("BLOCK 1 done | 4-bit:",not is_gpt2,"| bf16:",BF16,"| buyuk(7B/8B):",BIG,flush=True)

In [ ]:
# ===== BLOCK 2/3: training =====
from transformers import TrainingArguments, Trainer
BS = 2 if BIG else 8
args=TrainingArguments(output_dir=str(ROOT/"models"/f"{DATASET}_{MODEL.replace('/','_')}"),
    per_device_train_batch_size=BS, gradient_accumulation_steps=max(1,16//BS),
    num_train_epochs=EPOCHS, learning_rate=2e-4, bf16=BF16, fp16=(not BF16),
    gradient_checkpointing=(not is_gpt2), logging_steps=50, save_strategy="no", report_to=[], seed=SEED)
if not is_gpt2: model.config.use_cache=False
trainer=Trainer(model=model,args=args,train_dataset=DS(tr_sel))
t=time.time(); trainer.train(); print(f"BLOCK 2 done. training: {time.time()-t:.0f}s",flush=True)

In [ ]:
# ===== BLOCK 3/3: evaluate (full if test<=EVAL_CAP, else stratified subset) =====
from sklearn.metrics import accuracy_score, f1_score, classification_report
if len(te_idx)<=EVAL_CAP:
    eval_idx=te_idx; scope="tam test"
else:
    r2=np.random.default_rng(SEED); pick=[]
    for cls in np.unique(y_all[te_idx]):
        ids=te_idx[y_all[te_idx]==cls]; k=max(1,int(round(len(ids)*EVAL_CAP/len(te_idx))))
        pick+=r2.choice(ids,min(k,len(ids)),replace=False).tolist()
    eval_idx=np.array(sorted(pick)); scope=f"stratified subset ({len(eval_idx):,})"
print("evaluation:",scope,flush=True)
device=next(model.parameters()).device; model.eval()
bs=64; preds=[]; n=len(eval_idx); t=time.time()
for s in range(0,n,bs):
    js=eval_idx[s:s+bs]
    enc=tok(list(texts_all[js]),truncation=True,max_length=MAX_LEN,padding=True,return_tensors="pt").to(device)
    with torch.no_grad(), torch.autocast(device_type="cuda",dtype=torch.bfloat16,enabled=(device.type=="cuda")):
        logits=model(**enc).logits
    preds+=logits.argmax(-1).cpu().tolist()
    if (s//bs)%20==0: print(f"  ...{s:,}/{n:,}",flush=True)
pred=le.inverse_transform(np.array(preds)); yte=y_all[eval_idx]
acc=accuracy_score(yte,pred); mf1=f1_score(yte,pred,average="macro",labels=classes_all,zero_division=0)
wf1=f1_score(yte,pred,average="weighted",labels=classes_all,zero_division=0)
rp=classification_report(yte,pred,labels=classes_all,target_names=classes_all,output_dict=True,zero_division=0)
print(f"\n[{DATASET} / {MODEL}]  acc={acc:.4f}  macroF1={mf1:.4f}  weightedF1={wf1:.4f}  (eval {time.time()-t:.0f}s, {scope})",flush=True)
out=RES/"llm_report.json"; allr=json.load(open(out)) if out.exists() else {}
allr.setdefault(DATASET,{})[MODEL]={"acc":round(float(acc),4),"macro_f1":round(float(mf1),4),"weighted_f1":round(float(wf1),4),
    "eval_scope":scope,"train_cap":TRAIN_CAP,"epochs":EPOCHS,"n_train":int(len(tr_sel)),"n_eval":int(len(eval_idx)),
    "per_class_f1":{k:round(rp[k]['f1-score'],4) for k in classes_all}}
json.dump(allr,open(out,"w"),indent=2,ensure_ascii=False)
try:
    b=json.load(open(RES/"baseline_report.json"))[DATASET]["models"]
    print(f"\nKIYAS ({DATASET}) macro-F1:  {MODEL.split('/')[-1]}={mf1:.3f}  |  RF={b['random_forest']['macro_f1']}  XGB={b['xgboost']['macro_f1']}",flush=True)
except Exception: pass
print("BLOCK 3 done -> results/llm_report.json")